# Synchro Traffic Reports → Consolidated Excel

This notebook reads `.txt` exports from Trafficware Synchro (HCM 2000 Signalized report — **Lanes and Queues + HCM 2000 Signalized + Phases: Timings → Save to Text**) and produces a single consolidated `.xlsx` with one 8-row block per intersection.

**Works with Synchro 10 and 11** — tolerates extra/missing attribute rows and varying direction columns per intersection.

## How to use

1. Run the cells top-to-bottom (Runtime → Run all, or Shift+Enter each cell).
2. When prompted, upload one or more `.txt` Synchro exports.
3. The generated `.xlsx` downloads automatically.

Re-run the upload cell to process a different batch without restarting.


## 1. Install dependencies

In [ ]:
!pip install --quiet openpyxl


## 2. Load the parser and writer
These cells are auto-generated from `src/python/synchro_parser.py` and `src/python/synchro_writer.py`. Regenerate with `python scripts/build_notebook.py` after editing the modules.

In [ ]:
"""Parse Synchro 10/11 HCM 2000 text exports into a normalized structure.

The original VBA macro hard-coded row labels and column positions per Synchro
version. This parser keeps those labels discoverable but aligns every metric
row to its subsection's own direction-header row by tab column index, so the
same code reads Synchro 10 and Synchro 11 exports and tolerates optional or
extra rows.

Usage
-----
    from synchro_parser import parse_file
    for isx in parse_file("Sample/Synchro11/Sample2.txt"):
        print(isx.number, isx.name)
        print(isx.get_metric("Delay (s)", "EBL"))
"""

from __future__ import annotations

import json
import re
import sys
from dataclasses import dataclass, field
from pathlib import Path

SUBSECTIONS_OF_INTEREST: tuple[str, ...] = (
    "Lanes and Geometrics",
    "Queues",
    "HCM Signalized Intersection Capacity Analysis",
)

DIRECTION_HEADER_LABELS: tuple[str, ...] = ("Lane Group", "Movement")

_DIRECTION_RE = re.compile(r"^(EB|WB|NB|SB)[ULTR]$")
_INTERSECTION_HEADER_RE = re.compile(r"^\s*(\d+)\s*:\s*(.*)$")
_SYNCHRO_VERSION_RE = re.compile(r"Synchro\s+(\d+)\s+Report")
_LEADING_MARKERS_RE = re.compile(r"^[~#mc]+")
_TRAILING_CODES_RE = re.compile(r"(dl)$")


@dataclass
class Subsection:
    name: str
    directions: list[str] = field(default_factory=list)
    # metric label -> direction -> raw cell value (string)
    metrics: dict[str, dict[str, str]] = field(default_factory=dict)


@dataclass
class Intersection:
    number: int
    name: str
    date: str | None = None
    source_file: str | None = None
    synchro_version: int | None = None
    subsections: dict[str, Subsection] = field(default_factory=dict)

    def get_metric(
        self,
        label: str,
        direction: str,
        *,
        subsection: str | None = None,
    ) -> str | None:
        """Return the raw value for a metric in a direction, or None.

        If `subsection` is given, look only there. Otherwise search the
        subsections in the order `SUBSECTIONS_OF_INTEREST`.
        """
        if subsection is not None:
            sub = self.subsections.get(subsection)
            return sub.metrics.get(label, {}).get(direction) if sub else None
        for sub_name in SUBSECTIONS_OF_INTEREST:
            sub = self.subsections.get(sub_name)
            if sub is None:
                continue
            v = sub.metrics.get(label, {}).get(direction)
            if v is not None:
                return v
        return None


def clean_value(s: str) -> str:
    """Strip Synchro footnote markers from a cell value.

    Removes leading `~`, `#`, `m`, `c` (critical-lane / metered / volume-exceeds
    markers) and trailing `dl` (defacto left). Returns the trimmed string.
    """
    s = s.strip()
    if not s:
        return s
    s = _LEADING_MARKERS_RE.sub("", s)
    s = _TRAILING_CODES_RE.sub("", s)
    return s.strip()


def as_number(s: str) -> float | str:
    """Best-effort float coercion of a cleaned Synchro cell. Falls back to str."""
    s2 = clean_value(s)
    if not s2:
        return ""
    try:
        return float(s2)
    except ValueError:
        return s2


def _is_direction_code(s: str) -> bool:
    return bool(_DIRECTION_RE.match(s.strip()))


def parse_file(path: str | Path) -> list[Intersection]:
    p = Path(path)
    text = p.read_text(encoding="utf-8", errors="replace")
    if text.startswith("\ufeff"):
        text = text[1:]
    return parse_text(text, source_file=str(p))


def parse_text(text: str, source_file: str | None = None) -> list[Intersection]:
    lines = text.split("\n")
    intersections: dict[int, Intersection] = {}
    order: list[int] = []

    current_isx: Intersection | None = None
    current_sub: Subsection | None = None
    pending_subsection_name: str | None = None
    detected_version: int | None = None

    for raw in lines:
        line = raw.rstrip("\r")
        fields = line.split("\t")
        first = fields[0].strip() if fields else ""

        m = _SYNCHRO_VERSION_RE.search(line)
        if m:
            detected_version = int(m.group(1))
            if current_isx is not None and current_isx.synchro_version is None:
                current_isx.synchro_version = detected_version

        if first in SUBSECTIONS_OF_INTEREST:
            pending_subsection_name = first
            current_sub = None
            continue

        # Other subsection titles (e.g. "Timing Report, Sorted By Phase",
        # "HCM Unsignalized ..."): clear pending so we don't mis-attach the
        # next intersection header to a subsection we don't track.
        if first in ("Timing Report, Sorted By Phase",):
            pending_subsection_name = None
            current_sub = None
            continue

        if first == "Intersection Summary":
            current_sub = None
            continue

        if pending_subsection_name is not None and first and first[0].isdigit() and ":" in first:
            mh = _INTERSECTION_HEADER_RE.match(first)
            if mh:
                num = int(mh.group(1))
                name = mh.group(2).strip()
                date = fields[1].strip() if len(fields) > 1 and fields[1].strip() else None
                if num not in intersections:
                    intersections[num] = Intersection(
                        number=num,
                        name=name,
                        date=date,
                        source_file=source_file,
                        synchro_version=detected_version,
                    )
                    order.append(num)
                current_isx = intersections[num]
                sub_name = pending_subsection_name
                current_sub = current_isx.subsections.get(sub_name)
                if current_sub is None:
                    current_sub = Subsection(name=sub_name)
                    current_isx.subsections[sub_name] = current_sub
                pending_subsection_name = None
                continue

        if current_sub is None:
            continue

        if not current_sub.directions and first in DIRECTION_HEADER_LABELS:
            # Preserve column indexes: store directions as a dict[col_index -> code].
            dir_by_col: dict[int, str] = {}
            for i, cell in enumerate(fields[1:], start=1):
                c = cell.strip()
                if _is_direction_code(c):
                    dir_by_col[i] = c
            if dir_by_col:
                current_sub._dir_by_col = dir_by_col  # type: ignore[attr-defined]
                current_sub.directions = [dir_by_col[k] for k in sorted(dir_by_col)]
            continue

        if current_sub.directions and first:
            label = first
            dir_by_col: dict[int, str] = getattr(current_sub, "_dir_by_col", {})
            metric_row = current_sub.metrics.setdefault(label, {})
            for col_idx, direction in dir_by_col.items():
                if col_idx < len(fields):
                    cell = fields[col_idx].strip()
                    if cell:
                        metric_row[direction] = cell
            continue

    return [intersections[n] for n in order]


# ---------- CLI ----------

def _cli(argv: list[str]) -> int:
    if len(argv) < 2:
        print("usage: python synchro_parser.py <path-to-synchro-export.txt>", file=sys.stderr)
        return 2
    out = []
    for isx in parse_file(argv[1]):
        out.append(
            {
                "number": isx.number,
                "name": isx.name,
                "date": isx.date,
                "synchro_version": isx.synchro_version,
                "subsections": {
                    n: {"directions": s.directions, "metrics": s.metrics}
                    for n, s in isx.subsections.items()
                },
            }
        )
    json.dump(out, sys.stdout, indent=2)
    sys.stdout.write("\n")
    return 0


In [ ]:
"""Write a consolidated Synchro-report .xlsx from parsed intersections.

Mirrors the VBA macro's 8-rows-per-intersection block layout (see
src/vba/Main.bas around line 240) so the output is recognizable to existing
users. Each input folder becomes one sheet; intersections stack vertically.

Runtime dependency: openpyxl.

Usage
-----
Library:
    from synchro_parser import parse_file
    from synchro_writer import write_consolidated, parse_folder

    datasets = [parse_folder("path/to/2019 MD")]
    write_consolidated("out.xlsx", datasets)

CLI:
    python synchro_writer.py --out report.xlsx path/to/2019\\ MD path/to/2019\\ PM
"""

from __future__ import annotations

import argparse
import sys
from pathlib import Path

from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter


# (display label, subsection to look in) — order is the row order in each block.
DEFAULT_MOES: list[tuple[str, str]] = [
    ("Level of Service", "HCM Signalized Intersection Capacity Analysis"),
    ("Delay (s)", "HCM Signalized Intersection Capacity Analysis"),
    ("v/c Ratio", "HCM Signalized Intersection Capacity Analysis"),
    ("Queue Length 95th (m)", "Queues"),
    ("Storage Length (m)", "Lanes and Geometrics"),
]

BLOCK_ROWS = 1 + 1 + len(DEFAULT_MOES) + 1  # title + direction header + MOE rows + blank separator
TITLE_FILL = PatternFill("solid", fgColor="4472C4")  # Excel Accent1-ish
TITLE_FONT = Font(bold=True, color="FFFFFF")
HEADER_FONT = Font(bold=True)
CENTERED = Alignment(horizontal="center", vertical="center")


def parse_folder(folder: str | Path) -> tuple[str, list[Intersection]]:
    """Parse every .txt in `folder` and return (sheet_name, intersections).

    Sheet name is the folder's basename, uppercased to match the VBA convention.
    Intersections from multiple files are concatenated in filename order. No
    deduplication — same intersection number in two files produces two blocks.
    """
    p = Path(folder)
    if not p.is_dir():
        raise NotADirectoryError(p)
    all_isx: list[Intersection] = []
    for txt in sorted(p.glob("*.txt")):
        all_isx.extend(parse_file(txt))
    return p.name.upper(), all_isx


def write_consolidated(
    output_path: str | Path,
    datasets: list[tuple[str, list[Intersection]]],
    moes: list[tuple[str, str]] | None = None,
) -> None:
    """Write one sheet per dataset into a new .xlsx at `output_path`."""
    moes = moes or DEFAULT_MOES
    wb = Workbook()
    # openpyxl creates a default sheet; we'll reuse it for the first dataset.
    first = True
    for sheet_name, intersections in datasets:
        safe_name = _safe_sheet_name(sheet_name)
        ws = wb.active if first else wb.create_sheet()
        first = False
        ws.title = safe_name
        ws.column_dimensions["A"].width = 15.71
        ws.column_dimensions["B"].width = 21
        _write_sheet(ws, intersections, moes)
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    wb.save(output_path)


def _write_sheet(
    ws,
    intersections: list[Intersection],
    moes: list[tuple[str, str]],
) -> None:
    for idx, isx in enumerate(intersections):
        base = 1 + idx * BLOCK_ROWS  # openpyxl is 1-indexed

        # Row 0 of block: "Intersection N: Name" — spans direction columns
        directions = _directions_for_intersection(isx, moes)
        total_cols = 2 + max(1, len(directions))  # A (label) + B (row title) + direction cols
        title = f"Intersection {isx.number}: {isx.name}"
        ws.cell(row=base, column=1, value=title)
        if total_cols > 1:
            ws.merge_cells(
                start_row=base, start_column=1, end_row=base, end_column=total_cols
            )
        title_cell = ws.cell(row=base, column=1)
        title_cell.fill = TITLE_FILL
        title_cell.font = TITLE_FONT
        title_cell.alignment = CENTERED

        # Row 1: "" | "Direction" | EBL | EBT | ...
        ws.cell(row=base + 1, column=2, value="Direction").font = HEADER_FONT
        for j, d in enumerate(directions):
            c = ws.cell(row=base + 1, column=3 + j, value=d)
            c.font = HEADER_FONT
            c.alignment = CENTERED

        # Row 2: "Data Set" label in A
        ws.cell(row=base + 2, column=1, value="Data Set").font = HEADER_FONT

        # MOE rows: B column is the row title; C+ are values per direction.
        for k, (label, subsection) in enumerate(moes):
            r = base + 2 + k
            ws.cell(row=r, column=2, value=label).font = HEADER_FONT
            for j, d in enumerate(directions):
                raw = isx.get_metric(label, d, subsection=subsection)
                if raw is None:
                    continue
                value = as_number(raw)
                c = ws.cell(row=r, column=3 + j, value=value)
                c.alignment = CENTERED


def _directions_for_intersection(
    isx: Intersection, moes: list[tuple[str, str]]
) -> list[str]:
    """Union of direction codes across the subsections we'll read, preserving
    declaration order from each subsection. This matches the VBA behavior of
    taking the direction columns from `Lane Group`/`Movement` header rows,
    while tolerating subsections that omit some directions (e.g. Queues has
    no U-turns in Synchro 11 samples)."""
    seen: list[str] = []
    for _, subsection_name in moes:
        sub = isx.subsections.get(subsection_name)
        if sub is None:
            continue
        for d in sub.directions:
            if d not in seen:
                seen.append(d)
    return seen


_INVALID_SHEET_CHARS = set(r"[]:*?/\\")


def _safe_sheet_name(name: str) -> str:
    cleaned = "".join("_" if c in _INVALID_SHEET_CHARS else c for c in name)
    return cleaned[:31]  # Excel limit


def _cli(argv: list[str]) -> int:
    p = argparse.ArgumentParser(description="Consolidate Synchro exports into one .xlsx.")
    p.add_argument("folders", nargs="+", help="Input folder(s), each becomes one sheet.")
    p.add_argument("--out", "-o", required=True, help="Output .xlsx path.")
    args = p.parse_args(argv[1:])

    datasets = [parse_folder(f) for f in args.folders]
    total = sum(len(isx_list) for _, isx_list in datasets)
    if total == 0:
        print("no intersections parsed — check that folders contain .txt files", file=sys.stderr)
        return 1
    write_consolidated(args.out, datasets)
    print(f"wrote {args.out}: {len(datasets)} sheet(s), {total} intersection(s)")
    return 0


## 3. Upload your Synchro `.txt` files

Colab will open a file picker. Select one or more `.txt` exports — they are all treated as one dataset and written to a single sheet.

In [ ]:
# In Colab: opens a file picker. Running locally: falls back to a directory.
try:
    from google.colab import files as _colab_files
    _uploaded = _colab_files.upload()
    from pathlib import Path
    _workdir = Path("/content/synchro_input")
    _workdir.mkdir(exist_ok=True)
    for _name, _data in _uploaded.items():
        (_workdir / _name).write_bytes(_data)
    INPUT_DIR = _workdir
except ImportError:
    # Not in Colab — point at a local directory instead.
    from pathlib import Path
    INPUT_DIR = Path("./synchro_input")
    INPUT_DIR.mkdir(exist_ok=True)
    print(f"Not running in Colab. Drop .txt files into {INPUT_DIR.resolve()} and re-run the next cell.")

print(f"Input dir: {INPUT_DIR}")
print("Files:", sorted(p.name for p in INPUT_DIR.glob('*.txt')))


## 4. Parse and preview

In [ ]:
sheet_name, intersections = parse_folder(INPUT_DIR)
print(f"Parsed {len(intersections)} intersection(s) from '{sheet_name}'")
for isx in intersections[:5]:
    print(f"  #{isx.number}: {isx.name}  (Synchro {isx.synchro_version})")
if len(intersections) > 5:
    print(f"  ... +{len(intersections) - 5} more")


## 5. Write the consolidated `.xlsx` and download

In [ ]:
OUTPUT_NAME = "consolidated_report.xlsx"
write_consolidated(OUTPUT_NAME, [(sheet_name, intersections)])
print(f"Wrote {OUTPUT_NAME}")

try:
    from google.colab import files as _colab_files
    _colab_files.download(OUTPUT_NAME)
except ImportError:
    from pathlib import Path
    print(f"Saved to {Path(OUTPUT_NAME).resolve()}")


## Notes

- The MOEs written per intersection are set by `DEFAULT_MOES` in the writer cell — edit there to add/remove metrics.
- If a direction appears in one subsection but not another (e.g. Synchro 11 often omits U-turn columns from the Queues table), its column still appears in the output with blanks for the missing values.
- Footnote markers in raw Synchro values (`~`, `#`, `m`, `c`, trailing `dl`) are stripped when writing numeric cells.
